In [ ]:
import pandas as pd
import json
from gensim.models import Word2Vec
import re
import numpy as np
import nltk
from nltk.corpus import stopwords
from sklearn.metrics.pairwise import cosine_similarity



In [ ]:
# Cargar el dataset
df = pd.read_csv('/Users/franky/Desktop/Recipe recommender/datasets/recetas_ready.csv')

# Cargamos y limpiamos la lista del JSON
with open('/Users/franky/Desktop/Recipe recommender/datasets/tendencias.json', 'r') as f:
    # Quitamos espacios fantasmas que suelen quedar en los conteos
    ingredientes_estrella = [ing.strip().lower() for ing in json.load(f)]

print(f"ingredientes: {ingredientes_estrella}")

ingredientes: ['honey', 'fresh cilantro', 'avocado', 'lemon', 'red onion', 'ground cinnamon', 'vanilla extract', 'lime', 'pitted', 'lemon juice']


In [45]:
"""
En esta sección vamos a filtrar todas las recetas que contengan
al menos uno de los ingredientes tendencia que hemos identificado.
"""

# Verificamos el formato real de la columna
sample_ing = df['clean_ingredients'].iloc[0]
print(f"Tipo de dato en la columna: {type(sample_ing)}")
print(f"Ejemplo de contenido: {sample_ing}")

# Filtro usando un bucle explícito para evitar errores de tipo
def tiene_tendencia(lista_receta):
    # Si por alguna razón es un string y no una lista, lo convertimos
    if isinstance(lista_receta, str):
        # Intentamos limpiar caracteres de lista si vienen como string "[...]"
        lista_receta = lista_receta.replace('[', '').replace(']', '').replace("'", "").split(',')
    
    # Limpiamos cada ingrediente de la receta
    receta_normalizada = [str(i).strip().lower() for i in lista_receta]
    
    # Buscamos coincidencias
    for ing_top in ingredientes_estrella:
        if ing_top in receta_normalizada:
            return True
    return False

# Aplicamos
df_filtrado = df[df['clean_ingredients'].apply(tiene_tendencia)].copy()

print(f"\n--- RESULTADO ---")
print(f"Recetas originales: {len(df)}")
print(f"Recetas con ingredientes tendencia: {len(df_filtrado)}")

Tipo de dato en la columna: <class 'str'>
Ejemplo de contenido: ['strawberries', 'banana', 'yogurt', 'pineapple juice', 'white sugar', 'orange juice', 'milk']

--- RESULTADO ---
Recetas originales: 997
Recetas con ingredientes tendencia: 595


## Ahora pasamos a obtener las recetas mas similares 

In [ ]:
# Configuración de stopwords para limpieza de instrucciones
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Añadimos palabras que en instrucciones de cocina son muy comunes pero no definen la técnica
extra_stops = {'step', 'minutes', 'seconds', 'approx', 'recipe', 'using', 'prepared', 'place', 'put'}
stop_words.update(extra_stops)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/franky/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# Función de limpieza profunda para las instrucciones
def limpiar_direcciones_profundo(texto):
    # 1. Minúsculas y limpieza de caracteres
    texto = str(texto).lower()
    texto = re.sub(r'[^a-z\s]', ' ', texto)
    
    # 2. Tokenización y eliminación de stopwords
    palabras = texto.split()
    palabras_limpias = [w for w in palabras if w not in stop_words and len(w) > 2]
    
    return palabras_limpias # Devolvemos ya la lista de tokens para el modelo

# Aplicamos a nuestro DF filtrado
df_filtrado['tokens_directions'] = df_filtrado['directions'].apply(limpiar_direcciones_profundo)

In [48]:
# Entrenamos con los tokens limpios
model_nfn = Word2Vec(
    sentences=df_filtrado['tokens_directions'].tolist(),
    vector_size=100, 
    window=5, # Ventana más corta porque al quitar stopwords las palabras clave están más cerca
    min_count=1,
    sg=1
)

# Actualizamos los vectores de las recetas
df_filtrado['recipe_vector'] = df_filtrado['tokens_directions'].apply(
    lambda x: np.mean([model_nfn.wv[w] for w in x if w in model_nfn.wv], axis=0) 
    if x else np.zeros(100)
)

print("✅ Embeddings purificados: Stopwords eliminadas y ruido reducido.")

✅ Embeddings purificados: Stopwords eliminadas y ruido reducido.


In [49]:
# Aseguramos que las tendencias sean strings limpios
tendencias_limpias = [str(ing).strip().lower() for ing in ingredientes_estrella]

def buscador_infalible(fila):
    # Convertimos toda la fila de ingredientes a un solo string de texto
    texto_ingredientes = str(fila).lower()
    
    score = 0
    for ing in tendencias_limpias:
        if ing in texto_ingredientes:
            score += 1
    return score

# Aplicamos el conteo
df_filtrado['score_tendencia'] = df_filtrado['clean_ingredients'].apply(buscador_infalible)

# 2. Seleccionamos la mejor
id_ganador = df_filtrado['score_tendencia'].idxmax()
nombre_ganador = df_filtrado.loc[id_ganador, 'recipe_name']
puntaje_max = df_filtrado.loc[id_ganador, 'score_tendencia']

print(f"🏆 La receta líder es '{nombre_ganador}' con {puntaje_max} coincidencias.")

🏆 La receta líder es 'Savory Mango Guacamole' con 5 coincidencias.


In [ ]:
# Ahora vamos a usar el vector de esta receta para encontrar las más similares
# Extraemos el vector de nuestra receta estrella
vector_base = df_filtrado.loc[id_ganador, 'recipe_vector'].reshape(1, -1)
todos_los_vectores = np.stack(df_filtrado['recipe_vector'].values)

# Calculamos similitudes
sims = cosine_similarity(vector_base, todos_los_vectores).flatten()

# Obtenemos los 10 mejores (excluyendo la base)
indices_finales = sims.argsort()[-11:-1][::-1]
df_recomendadas = df_filtrado.iloc[indices_finales]

# --- RESULTADOS Y GUARDADO ---
print("\n✅ Recetas parecidas encontradas:")
print(df_recomendadas[['recipe_name', 'score_tendencia']])

# Guardamos la lista de nombres para el RAG
nombres_para_gemini = df_recomendadas['recipe_name'].tolist()

import json
with open('/Users/franky/Desktop/Recipe recommender/datasets/recetas_para_rag.json', 'w') as f:
    json.dump(nombres_para_gemini, f)

print("\n💾 Lista guardada en 'recetas_para_rag.json'. ¡Listo para el siguiente paso!")


✅ Recetas parecidas encontradas:
                       recipe_name  score_tendencia
666  Watermelon Fire and Ice Salsa                2
5                     Mango Relish                2
120            Avocado Mango Salsa                4
857              Fresh Mango Salsa                2
378                  Avocado Salad                3
488     Herb Watermelon Feta Salad                3
416       Quick and Easy Guacamole                4
442               Avocado Dressing                3
897              Spicy Mango Salad                2
316  Traditional Mexican Guacamole                3

💾 Lista guardada en 'recetas_para_rag.json'. ¡Listo para el siguiente paso!


In [51]:
# 1. Calculamos la similitud técnica de forma correcta
vector_base = df_filtrado.loc[id_ganador, 'recipe_vector'].reshape(1, -1)
vectores_recomendados = np.stack(df_recomendadas['recipe_vector'].values)

# Al usar flatten() aquí, obtenemos un array de 10 elementos (uno por cada receta recomendada)
similitudes_puras = cosine_similarity(vector_base, vectores_recomendados).flatten()

# 2. Creamos la tabla explicativa
analisis_scores = df_recomendadas[['recipe_name', 'score_tendencia']].copy()
analisis_scores['similitud_tecnica_%'] = (similitudes_puras * 100).round(2)

# 3. Ordenamos y mostramos
print("📊 ANÁLISIS DE RELEVANCIA Y SIMILITUD:")
display(analisis_scores.reset_index(drop=True))

📊 ANÁLISIS DE RELEVANCIA Y SIMILITUD:


,recipe_name,score_tendencia,similitud_tecnica_%
0,Watermelon Fire and Ice Salsa,2,99.739998
1,Mango Relish,2,99.690002
2,Avocado Mango Salsa,4,99.690002
3,Fresh Mango Salsa,2,99.589996
4,Avocado Salad,3,99.589996
5,Herb Watermelon Feta Salad,3,99.550003
6,Quick and Easy Guacamole,4,99.529999
7,Avocado Dressing,3,99.489998
8,Spicy Mango Salad,2,99.430000
9,Traditional Mexican Guacamole,3,99.389999


In [52]:
df.columns

Index(['recipe_name', 'prep_time_min', 'cook_time_min', 'total_time_min',
       'servings', 'ingredients', 'directions', 'rating', 'nutrition', 'date',
       'ingredients_list', 'clean_ingredients', 'w_sugar', 'num_ingredients'],
      dtype='str')